# 03 · Avaliação da geração (RAG de ponta a ponta)

Comparamos três sistemas com o **mesmo LLM** (Qwen3-4B-Instruct):

| Sistema | Contexto |
|---|---|
| **Sem RAG** | nenhum, o modelo responde "de memória" |
| **RAG + BM25** | top-5 chunks do BM25 |
| **RAG + melhor retriever** | top-5 chunks do melhor método do notebook 02 |

**Métricas**
- *Determinísticas* (sem LLM, 100% reprodutíveis): citação do artigo correto, citação "inventada"
  (artigo que não estava no contexto), recusa correta em perguntas fora do escopo e falsa recusa.
- *LLM-as-a-judge* (Qwen3-8B): **correção** (comparada à resposta de referência) e **fidelidade**
  (tudo o que foi dito está no contexto?), em escala de 1 a 5.
- *Validação do juiz*: você anota ~40 respostas à mão e medimos a concordância com o juiz.
  Sem isso, as métricas do juiz são só uma opinião de outro LLM.

**Entrada:** saída do notebook 02. **Configuração:** GPU T4 x2, Internet On. ⏱️ ~1h–1h30 de GPU.

In [ ]:
!pip install -q bitsandbytes

In [ ]:
import os, re, json, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

IN_KAGGLE = Path("/kaggle").exists()
OUT_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("saida_03")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"

# Depois de anotar à mão (seção 6), rode de novo com APENAS_ANALISE = True e SEM GPU:
# o notebook reaproveita as respostas já julgadas e só calcula a concordância.
APENAS_ANALISE = os.environ.get("APENAS_ANALISE") == "1"  # no Kaggle, basta trocar por True

GEN_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
JUDGE_MODEL_ID = "Qwen/Qwen3-8B"
TOP_CHUNKS = 5
N_ANOTACAO = 40
if SMOKE_TEST:
    GEN_MODEL_ID = JUDGE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"


def achar_arquivo(nome: str, obrigatorio: bool = True) -> Path | None:
    raizes = [Path("/kaggle/input")] if IN_KAGGLE else [Path("saida_02"), Path("saida_03"), Path(".")]
    for raiz in raizes:
        achados = sorted(raiz.glob(f"**/{nome}"))
        if achados:
            return achados[0]
    if obrigatorio:
        raise FileNotFoundError(f"{nome} não encontrado. Adicionou a saída do notebook 02 como input?")
    return None


chunks = pd.read_parquet(achar_arquivo("chunks.parquet")).set_index("chunk_id")
avaliacao = pd.read_parquet(achar_arquivo("avaliacao.parquet"))
recuperados = pd.read_parquet(achar_arquivo("recuperados.parquet"))
resumo_retrieval = json.load(open(achar_arquivo("resumo_retrieval.json")))
MELHOR = resumo_retrieval["melhor_metodo"]
print("Melhor retriever (notebook 02):", MELHOR)

if SMOKE_TEST:
    avaliacao = pd.concat([avaliacao[avaliacao.origem == "golden"].head(4),
                           avaliacao[avaliacao.origem == "fora_do_escopo"].head(2)])

## 1. Prompts

O formato de citação `[LEI, Art. N]` é fixo para podermos **verificar as citações com regex**,
sem depender de LLM. A frase de recusa também é fixa, pelo mesmo motivo.

In [ ]:
FRASE_RECUSA = "Não encontrei essa informação nos documentos consultados."

SYSTEM_RAG = f"""Você é um assistente que explica a legislação brasileira para cidadãos comuns.
Regras:
1. Responda SOMENTE com base nos trechos de lei fornecidos. Não use conhecimento externo.
2. Cite os artigos usados no formato [LEI, Art. N], por exemplo [CDC, Art. 49].
3. Se os trechos não responderem à pergunta, responda exatamente: "{FRASE_RECUSA}"
4. Seja claro e direto, em no máximo 5 frases. Isto não é aconselhamento jurídico."""

SYSTEM_SEM_RAG = f"""Você é um assistente que explica a legislação brasileira para cidadãos comuns.
Leis que você pode citar: CDC (Código de Defesa do Consumidor), LGPD, MCI (Marco Civil da Internet),
LI (Lei do Inquilinato) e CLT.
Regras:
1. Cite os artigos usados no formato [LEI, Art. N], por exemplo [CDC, Art. 49].
2. Se a pergunta não for sobre essas leis ou você não tiver certeza, responda exatamente: "{FRASE_RECUSA}"
3. Seja claro e direto, em no máximo 5 frases. Isto não é aconselhamento jurídico."""


def formatar_contexto(chunk_ids: list[str]) -> str:
    blocos = []
    for cid in chunk_ids[:TOP_CHUNKS]:
        c = chunks.loc[cid]
        blocos.append(f"[{c.lei}, Art. {c.artigo.split('#')[0]}] ({c.lei_nome})\n{c.texto}")
    return "\n\n---\n\n".join(blocos)


def prompt_rag(pergunta: str, contexto: str) -> str:
    return f"Trechos de lei:\n\n{contexto}\n\n=====\nPergunta: {pergunta}"

## 2. Geração das respostas

In [ ]:
import transformers
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"


def carregar_llm(model_id: str, quatro_bits: bool = False):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = {DTYPE_KW: torch.float16 if DEVICE == "cuda" else torch.float32}
    if quatro_bits and DEVICE == "cuda":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map={"": 0} if DEVICE == "cuda" else None, **kwargs)
    model.eval()
    return model, tok


@torch.inference_mode()
def gerar(model, tok, prompts: list[str], systems: list[str | None], max_new_tokens: int = 300,
          batch_size: int = 8) -> tuple[list[str], list[float]]:
    """Geração gulosa em lote. Devolve os textos e o tempo médio por item de cada lote."""
    textos = [
        tok.apply_chat_template(([{"role": "system", "content": s}] if s else []) + [{"role": "user", "content": p}],
                                tokenize=False, add_generation_prompt=True, enable_thinking=False)
        for p, s in zip(prompts, systems)
    ]
    ordem = sorted(range(len(textos)), key=lambda i: len(textos[i]))
    saidas, tempos = [None] * len(textos), [None] * len(textos)
    for i in tqdm(range(0, len(ordem), batch_size), desc="gerando"):
        idx = ordem[i:i + batch_size]
        enc = tok([textos[j] for j in idx], return_tensors="pt", padding=True).to(model.device)
        t0 = time.time()
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        dt = (time.time() - t0) / len(idx)
        for j, seq in zip(idx, out[:, enc["input_ids"].shape[1]:]):
            saidas[j] = tok.decode(seq, skip_special_tokens=True).strip()
            tempos[j] = dt
    return saidas, tempos


def extrair_json(texto: str):
    m = re.search(r"\{.*\}", texto, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None


def nota(valor) -> float:
    """Converte a nota do LLM para float em [1, 5]; qualquer outra coisa (texto, bool, null) vira NaN."""
    if isinstance(valor, bool):
        return float("nan")
    try:
        x = float(valor)
    except (TypeError, ValueError):
        return float("nan")
    return x if 1 <= x <= 5 else float("nan")


def liberar():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SISTEMAS = {"Sem RAG": None, "RAG + BM25": "BM25", f"RAG + {MELHOR}": MELHOR}
if MELHOR == "BM25":
    SISTEMAS.pop(f"RAG + {MELHOR}")

In [ ]:
if not APENAS_ANALISE:
    rec = recuperados.set_index(["qid", "metodo"])
    tarefas = []
    for sistema, metodo in SISTEMAS.items():
        for r in avaliacao.itertuples():
            if metodo is None:
                ctx_ids, prompt, system = [], f"Pergunta: {r.pergunta}", SYSTEM_SEM_RAG
            else:
                ctx_ids = list(rec.loc[(r.qid, metodo), "chunks"])[:TOP_CHUNKS]
                prompt, system = prompt_rag(r.pergunta, formatar_contexto(ctx_ids)), SYSTEM_RAG
            tarefas.append({"qid": r.qid, "sistema": sistema, "contexto_ids": ctx_ids, "prompt": prompt, "system": system})
    tarefas = pd.DataFrame(tarefas)
    print(f"{len(tarefas)} respostas a gerar")

    gen_model, gen_tok = carregar_llm(GEN_MODEL_ID)
    tarefas["resposta"], tarefas["seg_por_resposta"] = gerar(gen_model, gen_tok, tarefas.prompt.tolist(),
                                                             tarefas.system.tolist(), max_new_tokens=300, batch_size=16)
    del gen_model
    liberar()
    respostas = tarefas.merge(avaliacao, on="qid")
    respostas.drop(columns=["prompt", "system"]).to_parquet(OUT_DIR / "respostas.parquet", index=False)

## 3. Métricas determinísticas (citações e recusas)

In [ ]:
RE_CITACAO = re.compile(r"\[\s*(CDC|LGPD|MCI|LI|CLT)\s*,\s*Art\.?\s*(\d+)\s*[º°o]?\s*(?:-\s*([A-Z]{1,2}))?", re.IGNORECASE)
RE_RECUSA = re.compile(r"não encontrei essa informação", re.IGNORECASE)


def citacoes(texto: str) -> set[str]:
    return {f"{l.upper()}:{n}" + (f"-{s.upper()}" if s else "") for l, n, s in RE_CITACAO.findall(texto)}


def metricas_deterministicas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["citacoes"] = df.resposta.map(lambda t: sorted(citacoes(t)))
    df["recusou"] = df.resposta.str.contains(RE_RECUSA)
    df["tem_resposta"] = df.artigo_id.notna()
    df["citou_artigo_certo"] = [esp in cits if esp else np.nan for esp, cits in zip(df.artigo_id, df.citacoes)]
    artigos_ctx = [{chunks.loc[c].artigo_id.split("#")[0] for c in ids} for ids in df.contexto_ids]
    # Citação "inventada": artigo citado que não estava em nenhum trecho do contexto (só faz sentido com RAG)
    df["citacao_fora_do_contexto"] = [
        (len(set(cits) - ctx) > 0) if len(ids) else np.nan
        for cits, ctx, ids in zip(df.citacoes, artigos_ctx, df.contexto_ids)
    ]
    return df


if not APENAS_ANALISE:
    respostas = metricas_deterministicas(respostas)

## 4. LLM-as-a-judge

Um prompt só devolve as duas notas (economiza metade das chamadas). Perguntas fora do escopo não passam
pelo juiz: a métrica delas é a recusa, que já é determinística.

In [ ]:
PROMPT_JUIZ = """Você é um avaliador rigoroso de respostas sobre legislação brasileira.

Pergunta: {pergunta}
Resposta de referência (considerada correta): {referencia}
{bloco_contexto}
Resposta a avaliar:
\"\"\"{resposta}\"\"\"

Dê notas de 1 a 5:
- "correcao": a resposta concorda com a referência? 5 = equivalente; 3 = parcialmente correta ou incompleta;
  1 = errada, ou recusou responder.
- "fidelidade": {instrucao_fidelidade}

Responda apenas com JSON: {{"correcao": n, "fidelidade": n_ou_null, "justificativa": "uma frase"}}"""

FID_COM_CTX = "todas as afirmações da resposta estão apoiadas nos trechos acima? 5 = todas; 1 = inventa fatos."
FID_SEM_CTX = "use null (não há contexto para comparar)."


def prompt_juiz(r) -> str:
    tem_ctx = len(r.contexto_ids) > 0
    bloco = f"\nTrechos que o sistema recebeu:\n\"\"\"\n{formatar_contexto(list(r.contexto_ids))}\n\"\"\"\n" if tem_ctx else ""
    return PROMPT_JUIZ.format(pergunta=r.pergunta, referencia=r.resposta_referencia, bloco_contexto=bloco,
                              resposta=r.resposta, instrucao_fidelidade=FID_COM_CTX if tem_ctx else FID_SEM_CTX)


if not APENAS_ANALISE:
    # Recusar uma pergunta que tinha resposta vale correção 1 por definição: não gastamos o juiz com isso
    julgar = respostas[respostas.tem_resposta & ~respostas.recusou].copy()
    judge_model, judge_tok = carregar_llm(JUDGE_MODEL_ID, quatro_bits=True)
    saidas, _ = gerar(judge_model, judge_tok, [prompt_juiz(r) for r in julgar.itertuples()],
                      [None] * len(julgar), max_new_tokens=120, batch_size=8)
    del judge_model
    liberar()
    notas = [extrair_json(s) or {} for s in saidas]
    julgar["juiz_correcao"] = [nota(n.get("correcao")) for n in notas]
    julgar["juiz_fidelidade"] = [nota(n.get("fidelidade")) for n in notas]
    julgar["juiz_justificativa"] = [str(n.get("justificativa", "")) for n in notas]
    julgar.loc[julgar.contexto_ids.map(len) == 0, "juiz_fidelidade"] = np.nan  # sem contexto, não há fidelidade
    print(f"Saídas inválidas do juiz: {julgar.juiz_correcao.isna().mean():.1%}")
    respostas = respostas.merge(julgar[["qid", "sistema", "juiz_correcao", "juiz_fidelidade", "juiz_justificativa"]],
                                on=["qid", "sistema"], how="left")
    falsa_recusa = respostas.tem_resposta & respostas.recusou
    respostas.loc[falsa_recusa, "juiz_correcao"] = 1.0
    respostas.loc[falsa_recusa, "juiz_justificativa"] = "Recusou uma pergunta que tinha resposta nos documentos."
    respostas.to_parquet(OUT_DIR / "respostas_julgadas.parquet", index=False)
else:
    respostas = pd.read_parquet(achar_arquivo("respostas_julgadas.parquet"))

## 5. Resultados

In [ ]:
def resumir(df: pd.DataFrame) -> pd.DataFrame:
    linhas = []
    for sistema, g in df.groupby("sistema", sort=False):
        com, sem = g[g.tem_resposta], g[~g.tem_resposta]
        linhas.append({
            "Sistema": sistema,
            "Correção (1–5)": com.juiz_correcao.mean(),
            "% respostas corretas (nota ≥ 4)": (com.juiz_correcao.dropna() >= 4).mean(),
            "Fidelidade (1–5)": com.juiz_fidelidade.mean(),
            "% cita o artigo certo": com.citou_artigo_certo.astype(float).mean(),
            "% cita artigo fora do contexto": com.citacao_fora_do_contexto.astype(float).mean(),
            "% recusa correta (fora do escopo)": sem.recusou.mean(),
            "% falsa recusa (tinha resposta)": com.recusou.mean(),
            "Segundos por resposta": g.seg_por_resposta.mean(),
        })
    return pd.DataFrame(linhas).set_index("Sistema")


resultado = resumir(respostas)
resultado.round(3)

In [ ]:
# Golden vs sintéticas: as golden são a medida mais honesta (perguntas escritas por uma pessoa)
print(respostas[respostas.tem_resposta].pivot_table(index="sistema", columns="origem",
                                                    values="juiz_correcao", sort=False).round(2))

In [ ]:
import matplotlib.pyplot as plt

cols = ["% respostas corretas (nota ≥ 4)", "% cita o artigo certo", "% recusa correta (fora do escopo)"]
ax = resultado[cols].T.plot.bar(figsize=(9, 4.5), rot=0, width=0.75)
ax.set_ylim(0, 1)
ax.set_ylabel("proporção")
ax.set_title("Sem RAG vs RAG (mesmo LLM)")
ax.grid(axis="y", alpha=0.3)
ax.set_xticklabels(["Respostas corretas", "Cita o artigo certo", "Recusa fora do escopo"])
plt.tight_layout()
plt.savefig(OUT_DIR / "comparacao_sistemas.png", dpi=150)
plt.show()

In [ ]:
# Exemplos lado a lado (ótimos para colocar no README)
melhor_sistema = resultado["Correção (1–5)"].idxmax()
for qid in avaliacao[avaliacao.origem == "golden"].qid.head(3):
    sub = respostas[respostas.qid == qid]
    print(f"❓ {sub.pergunta.iloc[0]}  (esperado: {sub.artigo_id.iloc[0]})")
    for r in sub.itertuples():
        print(f"  • {r.sistema} [nota {r.juiz_correcao}]: {r.resposta[:300]}")
    print()

## 6. Validando o juiz com anotação humana

1. Baixe `para_anotar.csv` (aba *Output*), preencha a coluna `nota_humana` (1 a 5) **sem olhar a coluna do juiz**
   — ela fica escondida no arquivo `gabarito_juiz.csv`.
2. Salve como `anotacao_humana.csv` e suba como um Dataset do Kaggle.
3. Adicione o Dataset **e** a saída desta versão do notebook como input, mude `APENAS_ANALISE = True`
   e rode sem GPU.

In [ ]:
amostra_anot = (respostas[respostas.tem_resposta & (respostas.sistema != "Sem RAG")]
                .sample(min(N_ANOTACAO, int((respostas.tem_resposta & (respostas.sistema != "Sem RAG")).sum())),
                        random_state=0))
amostra_anot[["qid", "sistema", "pergunta", "resposta_referencia", "resposta"]].assign(nota_humana="") \
    .to_csv(OUT_DIR / "para_anotar.csv", index=False)
amostra_anot[["qid", "sistema", "juiz_correcao"]].to_csv(OUT_DIR / "gabarito_juiz.csv", index=False)

arquivo_humano = achar_arquivo("anotacao_humana.csv", obrigatorio=False)
concordancia = None
if arquivo_humano:
    from scipy.stats import spearmanr
    from sklearn.metrics import cohen_kappa_score

    humano = pd.read_csv(arquivo_humano).dropna(subset=["nota_humana"])
    comp = humano.merge(respostas[["qid", "sistema", "juiz_correcao"]], on=["qid", "sistema"]).dropna()
    h, j = comp.nota_humana.astype(int), comp.juiz_correcao.astype(int)
    concordancia = {
        "n": len(comp),
        "spearman": float(spearmanr(h, j).statistic),
        "kappa_ponderado": float(cohen_kappa_score(h, j, weights="quadratic")),
        "concordancia_binaria (correta = nota ≥ 4)": float(((h >= 4) == (j >= 4)).mean()),
    }
    print(concordancia)
    # Onde o juiz mais diverge de você? Isso diz se ele é "bonzinho" ou "rigoroso" demais.
    print("Média humano - juiz:", round(float((h - j).mean()), 2))
else:
    print("Ainda sem anotação humana: preencha para_anotar.csv e volte aqui.")

## 7. Salvando o resumo

In [ ]:
resumo = {
    "gerador": GEN_MODEL_ID, "juiz": JUDGE_MODEL_ID, "retriever": MELHOR, "top_chunks": TOP_CHUNKS,
    "n_perguntas": int(len(avaliacao)), "resultado": resultado.reset_index().to_dict(orient="records"),
    "concordancia_juiz_humano": concordancia,
}
json.dump(resumo, open(OUT_DIR / "resumo_geracao.json", "w"), ensure_ascii=False, indent=2, default=float)
resultado.to_csv(OUT_DIR / "metricas_geracao.csv")
print(json.dumps(resumo, ensure_ascii=False, indent=2, default=float)[:2000])